# pngmodel — Logo Refiner (Real-ESRGAN fine-tune) on Colab

Fine-tunes Real-ESRGAN on **full-color** logos (gilbarbara/logos), then upscales a logo to a
high-res PNG + SVG.

**Before running:** `Runtime -> Change runtime type -> GPU` (free tier gives a T4, 16 GB).

Run the cells top to bottom. The whole dataset is rebuilt on Colab in ~1 minute, so there is
nothing to upload.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## 2. Install dependencies

`basicsr` needs `numpy<2`. If Colab had numpy 2.x, **restart the runtime when prompted**
(Runtime -> Restart), then continue from cell 3 onward (skip this cell).

In [ ]:
!pip install -q "numpy<2" basicsr==1.4.2 realesrgan==0.3.0 tb-nightly vtracer cairosvg
print("\nIf numpy was downgraded, restart the runtime now, then skip this cell.")

## 3. Write the helper scripts

In [ ]:
%%writefile render_svgs.py
# Render SVG logos -> square white PNGs (aspect ratio preserved)
import io, sys, glob, os
from PIL import Image
import cairosvg

inp, out = sys.argv[1], sys.argv[2]
size = int(sys.argv[3]) if len(sys.argv) > 3 else 512
os.makedirs(out, exist_ok=True)
svgs = sorted(glob.glob(os.path.join(inp, '**', '*.svg'), recursive=True))
ok = 0
for s in svgs:
    try:
        b = cairosvg.svg2png(url=s, output_width=size)
        im = Image.open(io.BytesIO(b)).convert('RGBA')
        if im.height > size:
            b = cairosvg.svg2png(url=s, output_height=size)
            im = Image.open(io.BytesIO(b)).convert('RGBA')
        canvas = Image.new('RGBA', (size, size), (255, 255, 255, 255))
        canvas.alpha_composite(im, (max((size - im.width)//2, 0), max((size - im.height)//2, 0)))
        canvas.convert('RGB').save(os.path.join(out, os.path.splitext(os.path.basename(s))[0] + '.png'))
        ok += 1
    except Exception as e:
        print('skip', os.path.basename(s), e)
print('rendered', ok, 'of', len(svgs))


In [ ]:
%%writefile prepare_data.py
# Build HR dataset (+ multiscale) and a meta_info file for Real-ESRGAN
import sys, os, glob
import cv2, numpy as np
from PIL import Image

inp, out, meta = sys.argv[1], sys.argv[2], sys.argv[3]
minsize = int(sys.argv[4]) if len(sys.argv) > 4 else 256
factors = [1.0, 0.75, 0.5, 1/3]
os.makedirs(out, exist_ok=True)
os.makedirs(os.path.dirname(meta), exist_ok=True)
files = [f for f in glob.glob(os.path.join(inp, '**', '*'), recursive=True)
         if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp'))]
written = []
for f in files:
    try:
        img = Image.open(f).convert('RGBA')
        bg = Image.new('RGBA', img.size, (255, 255, 255, 255))
        rgb = Image.alpha_composite(bg, img).convert('RGB')
        a = cv2.cvtColor(np.array(rgb), cv2.COLOR_RGB2BGR)
    except Exception as e:
        print('skip', f, e); continue
    stem = os.path.splitext(os.path.basename(f))[0]
    h, w = a.shape[:2]
    for i, fc in enumerate(factors):
        nh, nw = int(h*fc), int(w*fc)
        if min(nh, nw) < minsize:
            continue
        r = a if fc == 1.0 else cv2.resize(a, (nw, nh), interpolation=cv2.INTER_AREA)
        name = stem + ('' if (fc == 1.0 and i == 0) else '_s%03d' % int(fc*100)) + '.png'
        cv2.imwrite(os.path.join(out, name), r)
        written.append(name)
open(meta, 'w').write('\n'.join(written) + '\n')
print('prepared', len(written), 'images ; meta_info ->', meta)


In [ ]:
%%writefile train.py
# Thin launcher: patch torchvision, register Real-ESRGAN, run BasicSR training
import sys, os, types
def patch():
    try:
        import torchvision.transforms.functional_tensor  # noqa
    except ModuleNotFoundError:
        import torchvision.transforms.functional as F
        m = types.ModuleType('torchvision.transforms.functional_tensor')
        m.rgb_to_grayscale = F.rgb_to_grayscale
        sys.modules['torchvision.transforms.functional_tensor'] = m
patch()
import realesrgan.archs, realesrgan.data, realesrgan.models  # noqa
from basicsr.train import train_pipeline
cfg = sys.argv[1]
sys.argv = [sys.argv[0], '-opt', cfg, '--auto_resume']
train_pipeline(os.getcwd())


## 4. Build the colored dataset (clone -> render -> prepare)

In [ ]:
!git clone --depth 1 https://github.com/gilbarbara/logos.git _srcsets/gilbarbara-logos
!python render_svgs.py _srcsets/gilbarbara-logos/logos data/logos_hr_color 512
!python prepare_data.py data/logos_hr_color data/logos_hr_color_prepared data/meta_info/meta_info_logos_color.txt 256

## 5. Download the pretrained base weights

In [ ]:
!mkdir -p weights
!wget -q -O weights/RealESRNet_x4plus.pth https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.1/RealESRNet_x4plus.pth
!wget -q -O weights/RealESRGAN_x4plus.pth https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth
!ls -lh weights

## 6. Write the configs (tuned for a T4, 16 GB)

Bigger batches than the 8 GB local configs: Stage 1 batch 12, Stage 2 batch 8.
`queue_size: 192` stays divisible by both. Stage-2 losses are tuned for flat logo art
(higher L1, lower perceptual + GAN) so it won't hallucinate texture.

In [ ]:
%%writefile finetune_realesrnet_logos.yml
name: finetune_RealESRNetx4plus_logos
model_type: RealESRNetModel
scale: 4
num_gpu: 1
manual_seed: 0
gt_usm: True
resize_prob: [0.2, 0.7, 0.1]
resize_range: [0.15, 1.5]
gaussian_noise_prob: 0.5
noise_range: [1, 30]
poisson_scale_range: [0.05, 3]
gray_noise_prob: 0.4
jpeg_range: [30, 95]
second_blur_prob: 0.8
resize_prob2: [0.3, 0.4, 0.3]
resize_range2: [0.3, 1.2]
gaussian_noise_prob2: 0.5
noise_range2: [1, 25]
poisson_scale_range2: [0.05, 2.5]
gray_noise_prob2: 0.4
jpeg_range2: [30, 95]
gt_size: 256
queue_size: 192
datasets:
  train:
    name: logos
    type: RealESRGANDataset
    dataroot_gt: data/logos_hr_color_prepared
    meta_info: data/meta_info/meta_info_logos_color.txt
    io_backend:
      type: disk
    blur_kernel_size: 21
    kernel_list: ['iso', 'aniso', 'generalized_iso', 'generalized_aniso', 'plateau_iso', 'plateau_aniso']
    kernel_prob: [0.45, 0.25, 0.12, 0.03, 0.12, 0.03]
    sinc_prob: 0.1
    blur_sigma: [0.2, 3]
    betag_range: [0.5, 4]
    betap_range: [1, 2]
    blur_kernel_size2: 21
    kernel_list2: ['iso', 'aniso', 'generalized_iso', 'generalized_aniso', 'plateau_iso', 'plateau_aniso']
    kernel_prob2: [0.45, 0.25, 0.12, 0.03, 0.12, 0.03]
    sinc_prob2: 0.1
    blur_sigma2: [0.2, 1.5]
    betag_range2: [0.5, 4]
    betap_range2: [1, 2]
    final_sinc_prob: 0.8
    gt_size: 256
    use_hflip: True
    use_rot: False
    use_shuffle: true
    num_worker_per_gpu: 2
    batch_size_per_gpu: 12
    dataset_enlarge_ratio: 1
    prefetch_mode: ~
network_g:
  type: RRDBNet
  num_in_ch: 3
  num_out_ch: 3
  num_feat: 64
  num_block: 23
  num_grow_ch: 32
path:
  pretrain_network_g: weights/RealESRNet_x4plus.pth
  param_key_g: params_ema
  strict_load_g: true
  resume_state: ~
train:
  ema_decay: 0.999
  optim_g:
    type: Adam
    lr: !!float 2e-4
    weight_decay: 0
    betas: [0.9, 0.99]
  scheduler:
    type: MultiStepLR
    milestones: [40000]
    gamma: 0.5
  total_iter: 20000
  warmup_iter: -1
  pixel_opt:
    type: L1Loss
    loss_weight: 1.0
    reduction: mean
logger:
  print_freq: 100
  save_checkpoint_freq: !!float 2e3
  use_tb_logger: true
  wandb:
    project: ~
    resume_id: ~
dist_params:
  backend: nccl
  port: 29500


In [ ]:
%%writefile finetune_realesrgan_logos.yml
name: finetune_RealESRGANx4plus_logos
model_type: RealESRGANModel
scale: 4
num_gpu: 1
manual_seed: 0
l1_gt_usm: True
percep_gt_usm: True
gan_gt_usm: False
resize_prob: [0.2, 0.7, 0.1]
resize_range: [0.15, 1.5]
gaussian_noise_prob: 0.5
noise_range: [1, 30]
poisson_scale_range: [0.05, 3]
gray_noise_prob: 0.4
jpeg_range: [30, 95]
second_blur_prob: 0.8
resize_prob2: [0.3, 0.4, 0.3]
resize_range2: [0.3, 1.2]
gaussian_noise_prob2: 0.5
noise_range2: [1, 25]
poisson_scale_range2: [0.05, 2.5]
gray_noise_prob2: 0.4
jpeg_range2: [30, 95]
gt_size: 256
queue_size: 192
datasets:
  train:
    name: logos
    type: RealESRGANDataset
    dataroot_gt: data/logos_hr_color_prepared
    meta_info: data/meta_info/meta_info_logos_color.txt
    io_backend:
      type: disk
    blur_kernel_size: 21
    kernel_list: ['iso', 'aniso', 'generalized_iso', 'generalized_aniso', 'plateau_iso', 'plateau_aniso']
    kernel_prob: [0.45, 0.25, 0.12, 0.03, 0.12, 0.03]
    sinc_prob: 0.1
    blur_sigma: [0.2, 3]
    betag_range: [0.5, 4]
    betap_range: [1, 2]
    blur_kernel_size2: 21
    kernel_list2: ['iso', 'aniso', 'generalized_iso', 'generalized_aniso', 'plateau_iso', 'plateau_aniso']
    kernel_prob2: [0.45, 0.25, 0.12, 0.03, 0.12, 0.03]
    sinc_prob2: 0.1
    blur_sigma2: [0.2, 1.5]
    betag_range2: [0.5, 4]
    betap_range2: [1, 2]
    final_sinc_prob: 0.8
    gt_size: 256
    use_hflip: True
    use_rot: False
    use_shuffle: true
    num_worker_per_gpu: 2
    batch_size_per_gpu: 8
    dataset_enlarge_ratio: 1
    prefetch_mode: ~
network_g:
  type: RRDBNet
  num_in_ch: 3
  num_out_ch: 3
  num_feat: 64
  num_block: 23
  num_grow_ch: 32
network_d:
  type: UNetDiscriminatorSN
  num_in_ch: 3
  num_feat: 64
  skip_connection: True
path:
  pretrain_network_g: experiments/finetune_RealESRNetx4plus_logos/models/net_g_20000.pth
  param_key_g: params_ema
  strict_load_g: true
  pretrain_network_d: ~
  param_key_d: params
  strict_load_d: true
  resume_state: ~
train:
  ema_decay: 0.999
  optim_g:
    type: Adam
    lr: !!float 1e-4
    weight_decay: 0
    betas: [0.9, 0.99]
  optim_d:
    type: Adam
    lr: !!float 1e-4
    weight_decay: 0
    betas: [0.9, 0.99]
  scheduler:
    type: MultiStepLR
    milestones: [40000]
    gamma: 0.5
  total_iter: 20000
  warmup_iter: -1
  pixel_opt:
    type: L1Loss
    loss_weight: 1.5
    reduction: mean
  perceptual_opt:
    type: PerceptualLoss
    layer_weights:
      'conv1_2': 0.1
      'conv2_2': 0.1
      'conv3_4': 1
      'conv4_4': 1
      'conv5_4': 1
    vgg_type: vgg19
    use_input_norm: true
    perceptual_weight: !!float 0.5
    style_weight: 0
    range_norm: false
    criterion: l1
  gan_opt:
    type: GANLoss
    gan_type: vanilla
    real_label_val: 1.0
    fake_label_val: 0.0
    loss_weight: !!float 3e-2
  net_d_iters: 1
  net_d_init_iters: 0
logger:
  print_freq: 100
  save_checkpoint_freq: !!float 2e3
  use_tb_logger: true
  wandb:
    project: ~
    resume_id: ~
dist_params:
  backend: nccl
  port: 29500


## 7. Stage 1 — RealESRNet (L1 base)

~1.5-2 h on a T4. Checkpoints land in `experiments/`. `--auto_resume` is on, so if Colab
disconnects, just re-run this cell to continue.

In [ ]:
!python train.py finetune_realesrnet_logos.yml

## 8. Stage 2 — RealESRGAN (GAN). Run only after Stage 1 finishes.

In [ ]:
!python train.py finetune_realesrgan_logos.yml

## 9. Save your trained model to Google Drive

Colab wipes local disk when the session ends — copy the results out.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pngmodel_colab
!cp -r experiments /content/drive/MyDrive/pngmodel_colab/
print('Saved experiments/ to Drive -> MyDrive/pngmodel_colab')

## 10. Test it — refine a logo (PNG + SVG)

Upload a low-quality logo when prompted, then run.

In [ ]:
%%writefile vectorize.py
import sys, vtracer
vtracer.convert_image_to_svg_py(sys.argv[1], sys.argv[2], colormode='color',
    hierarchical='stacked', mode='spline', filter_speckle=4, color_precision=6,
    layer_difference=16, corner_threshold=60, length_threshold=4.0,
    max_iterations=10, splice_threshold=45, path_precision=8)
print('SVG ->', sys.argv[2])


In [ ]:
import sys, types
def patch():
    try:
        import torchvision.transforms.functional_tensor  # noqa
    except ModuleNotFoundError:
        import torchvision.transforms.functional as F
        m = types.ModuleType('torchvision.transforms.functional_tensor')
        m.rgb_to_grayscale = F.rgb_to_grayscale
        sys.modules['torchvision.transforms.functional_tensor'] = m
patch()
import cv2, glob
from google.colab import files
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
up = files.upload()
src = list(up.keys())[0]
model_path = 'experiments/finetune_RealESRGANx4plus_logos/models/net_g_20000.pth'
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
ups = RealESRGANer(scale=4, model_path=model_path, model=model, tile=0, half=True)
img = cv2.imread(src, cv2.IMREAD_UNCHANGED)
out, _ = ups.enhance(img, outscale=4)
cv2.imwrite('refined.png', out)
import os
os.system('python vectorize.py refined.png refined.svg')
from IPython.display import Image as IPImage, display
display(IPImage('refined.png'))
files.download('refined.png')
files.download('refined.svg')